# PDCH Text Modality — Gentler Models (v4): LoRA + Frozen-MLP

## 0. Shared config & folds

In [ ]:
from pathlib import Path
import numpy as np, json, time, os
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, classification_report

# --- self-healing Drive mount ---
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

PROC_DIR = Path('/content/drive/MyDrive/PDCH/processed_v1')
EMB_NPZ  = PROC_DIR / 'embeddings' / 'pdch_text_emb_full_dialogue.npz'
OUT_DIR  = PROC_DIR / 'gentle_v4'; OUT_DIR.mkdir(parents=True, exist_ok=True)
SEED=42; N_FOLDS=5
PROBE_BASELINE=0.449; FULL_FT_BASELINE=0.386

# verify the file is reachable before loading (clearer error if not)
assert EMB_NPZ.exists(), f"Not reachable: {EMB_NPZ}\nDrive may be unmounted — re-run this cell, or re-run v2 to regenerate."

d=np.load(EMB_NPZ, allow_pickle=True)
ids=d['ids']; X=d['X'].astype('float32'); y=d['y3'].astype(int)
print('embeddings:', X.shape, '| sev3 dist:', np.bincount(y))

skf=StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds=list(skf.split(np.arange(len(ids)), y))
for k,(tr,te) in enumerate(folds):
    assert set(ids[tr]).isdisjoint(set(ids[te])), f'leak {k}'
print(f'{N_FOLDS} patient-level folds (seed={SEED}) — same as v2/v3, disjoint verified')

In [ ]:
import glob, os
print("Drive mounted?", os.path.exists('/content/drive/MyDrive'), "\n")

print("Anywhere in Drive named pdch_text_emb*:")
for h in glob.glob('/content/drive/**/pdch_text_emb*', recursive=True):
    print("   ", h)

print("\nAnywhere on local disk (/content, non-Drive):")
for h in glob.glob('/content/pdch_text_emb*') + glob.glob('/content/**/pdch_text_emb*', recursive=True):
    if '/drive/' not in h: print("   ", h)

print("\nWhat's actually inside processed_v1:")
base = '/content/drive/MyDrive/PDCH/processed_v1'
if os.path.exists(base):
    for p in sorted(glob.glob(base + '/**', recursive=True)):
        print("   ", p)
else:
    print("   processed_v1 does NOT exist in Drive")

In [ ]:
from pathlib import Path
import numpy as np, json, time
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, classification_report

PROC_DIR = Path('/content/drive/MyDrive/PDCH/processed_v1')
EMB_NPZ  = PROC_DIR / 'embeddings' / 'pdch_text_emb_full_dialogue.npz'   # from v2
OUT_DIR  = PROC_DIR / 'gentle_v4'; OUT_DIR.mkdir(parents=True, exist_ok=True)
SEED=42; N_FOLDS=5
PROBE_BASELINE=0.449; FULL_FT_BASELINE=0.386

# load v2 frozen embeddings + labels
d=np.load(EMB_NPZ, allow_pickle=True)
ids=d['ids']; X=d['X'].astype('float32'); y=d['y3'].astype(int)
print('embeddings:', X.shape, '| sev3 dist:', np.bincount(y))

skf=StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds=list(skf.split(np.arange(len(ids)), y))
for k,(tr,te) in enumerate(folds):
    assert set(ids[tr]).isdisjoint(set(ids[te])), f'leak {k}'
print(f'{N_FOLDS} patient-level folds (seed={SEED}) — same as v2/v3, disjoint verified')

## Section A — Frozen encoder + MLP head

In [ ]:
import torch, torch.nn as nn
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(SEED); np.random.seed(SEED)

class MLPHead(nn.Module):
    def __init__(self, d_in, hidden=256, n_cls=3, p=0.3):
        super().__init__()
        self.net=nn.Sequential(nn.LayerNorm(d_in), nn.Linear(d_in,hidden), nn.GELU(),
                               nn.Dropout(p), nn.Linear(hidden,n_cls))
    def forward(self,x): return self.net(x)

def class_w(yt):
    c=np.bincount(yt,minlength=3).astype(float); c[c==0]=1
    return torch.tensor(c.sum()/(3*c), dtype=torch.float32, device=DEVICE)

def run_mlp_fold(tr, te, epochs=120, patience=15, lr=1e-3):
    tr2,va=train_test_split(tr, test_size=0.15, stratify=y[tr], random_state=SEED)
    Xtr=torch.tensor(X[tr2],device=DEVICE); ytr=torch.tensor(y[tr2],device=DEVICE)
    Xva=torch.tensor(X[va],device=DEVICE);  Xte=torch.tensor(X[te],device=DEVICE)
    m=MLPHead(X.shape[1]).to(DEVICE); opt=torch.optim.AdamW(m.parameters(),lr=lr,weight_decay=1e-2)
    lossf=nn.CrossEntropyLoss(weight=class_w(y[tr2]))
    best,beststate,bad=-1,None,0
    for ep in range(epochs):
        m.train(); opt.zero_grad(); loss=lossf(m(Xtr),ytr); loss.backward(); opt.step()
        m.eval()
        with torch.no_grad(): vp=m(Xva).argmax(1).cpu().numpy()
        vf=f1_score(y[va],vp,average='macro')
        if vf>best: best,beststate,bad=vf,{k:v.cpu().clone() for k,v in m.state_dict().items()},0
        else:
            bad+=1
            if bad>=patience: break
    m.load_state_dict(beststate); m.eval()
    with torch.no_grad(): return m(Xte).argmax(1).cpu().numpy()

mlp_f1=[]; oof_mlp=np.full(len(ids),-1)
for k,(tr,te) in enumerate(folds):
    pred=run_mlp_fold(tr,te); oof_mlp[te]=pred
    f=f1_score(y[te],pred,average='macro'); mlp_f1.append(f)
    print(f'  [MLP] fold {k+1} macro-F1={f:.3f}')
print(f'\nfrozen+MLP: {np.mean(mlp_f1):.3f} ± {np.std(mlp_f1):.3f}')

## Section B — LoRA fine-tune (q,v adapters)

In [ ]:
# !pip install -q peft transformers
import pandas as pd, re
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
from peft import LoraConfig, get_peft_model

ENCODER_NAME='hfl/chinese-roberta-wwm-ext'
MIN_CHARS=4; MAX_CHARS=400; MAX_TOKENS=512; CHUNK_BATCH=16
LORA_R=8; LORA_ALPHA=16; LORA_DROPOUT=0.1; LR_LORA=3e-4; EPOCHS_LORA=8; PATIENCE_LORA=2

# rebuild full_dialogue chunks per patient (same as v2/v3)
turns=pd.read_parquet(PROC_DIR/'pdch_turns.parquet')
turns=turns[turns.patient_id.isin(set(ids))]
def merge_consec(g):
    out=[]
    for r,t in zip(g.role,g.text):
        if out and out[-1][0]==r: out[-1]=(r,out[-1][1]+' '+t)
        else: out.append((r,t))
    return out
def build_fd(g):
    g=g.sort_values('turn_idx'); pref={'patient':'患者：','doctor':'医生：'}
    return [pref[r]+t for r,t in merge_consec(g) if len(t)>=MIN_CHARS]
def pack(units,mx=MAX_CHARS):
    ch,cur=[],''
    for u in units:
        if len(u)>mx:
            if cur: ch.append(cur); cur=''
            for i in range(0,len(u),mx): ch.append(u[i:i+mx])
            continue
        if len(cur)+len(u)+1<=mx: cur=(cur+'\n'+u) if cur else u
        else: ch.append(cur); cur=u
    if cur: ch.append(cur)
    return ch if ch else ['']
pchunks={pid:pack(build_fd(g)) for pid,g in turns.groupby('patient_id')}
id2row={pid:i for i,pid in enumerate(ids)}
tokenizer=AutoTokenizer.from_pretrained(ENCODER_NAME)

class CDS(Dataset):
    def __init__(self,pidx):
        self.items=[(ch,pi) for pi in pidx for ch in pchunks[ids[pi]]]
    def __len__(self): return len(self.items)
    def __getitem__(self,i): return self.items[i]
def coll(b):
    enc=tokenizer([x[0] for x in b],padding=True,truncation=True,max_length=MAX_TOKENS,return_tensors='pt')
    return enc, torch.tensor([x[1] for x in b])

class LoRANet(nn.Module):
    def __init__(self):
        super().__init__()
        base=AutoModel.from_pretrained(ENCODER_NAME)
        cfg=LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                       target_modules=['query','value'], bias='none')
        self.enc=get_peft_model(base,cfg)
        self.drop=nn.Dropout(0.2); self.head=nn.Linear(base.config.hidden_size,3)
    def forward(self,enc):
        o=self.enc(**enc).last_hidden_state; m=enc['attention_mask'].unsqueeze(-1).float()
        e=(o*m).sum(1)/m.sum(1).clamp(min=1e-9); return self.head(self.drop(e))

def cw_lora(idx): return {pi:1.0/len(pchunks[ids[pi]]) for pi in idx}
@torch.no_grad()
def pool_predict(model,pidx):
    model.eval(); dl=DataLoader(CDS(pidx),batch_size=CHUNK_BATCH,collate_fn=coll)
    acc={pi:np.zeros(3) for pi in pidx}
    for enc,pi in dl:
        enc={k:v.to(DEVICE) for k,v in enc.items()}
        p=torch.softmax(model(enc),-1).cpu().numpy()
        for j,x in enumerate(pi.numpy()): acc[x]+=p[j]
    return np.array([acc[pi].argmax() for pi in pidx])

def run_lora_fold(tr,te):
    tr2,va=train_test_split(tr,test_size=0.15,stratify=y[tr],random_state=SEED)
    model=LoRANet().to(DEVICE)
    opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=LR_LORA)
    ce=nn.CrossEntropyLoss(weight=class_w(y[tr2])); wmap=cw_lora(tr2)
    dl=DataLoader(CDS(tr2),batch_size=CHUNK_BATCH,shuffle=True,collate_fn=coll)
    best,beststate,bad=-1,None,0
    for ep in range(EPOCHS_LORA):
        model.train()
        for enc,pi in dl:
            enc={k:v.to(DEVICE) for k,v in enc.items()}
            yb=torch.tensor(y[pi.numpy()],device=DEVICE)
            w=torch.tensor([wmap[int(x)] for x in pi],dtype=torch.float32,device=DEVICE)
            loss=(ce(model(enc),yb)*w).sum()/w.sum()
            opt.zero_grad(); loss.backward(); opt.step()
        vf=f1_score(y[va],pool_predict(model,va),average='macro')
        if vf>best: best,beststate,bad=vf,{k:v.cpu().clone() for k,v in model.state_dict().items()},0
        else:
            bad+=1
            if bad>=PATIENCE_LORA: break
    model.load_state_dict(beststate)
    return pool_predict(model,te)

lora_f1=[]; oof_lora=np.full(len(ids),-1)
for k,(tr,te) in enumerate(folds):
    t0=time.time(); pred=run_lora_fold(tr,te); oof_lora[te]=pred
    f=f1_score(y[te],pred,average='macro'); lora_f1.append(f)
    print(f'  [LoRA] fold {k+1} macro-F1={f:.3f} ({time.time()-t0:.0f}s)')
    if DEVICE=='cuda': torch.cuda.empty_cache()
print(f'\nLoRA: {np.mean(lora_f1):.3f} ± {np.std(lora_f1):.3f}')

In [ ]:
# Import the userdata module
from google.colab import userdata

# Load the HF_TOKEN from Colab secrets
os.environ['HF_TOKEN'] = userdata.get('Bert')

print("HF_TOKEN loaded from Colab secrets. Please restart your Colab runtime for this to take full effect.")

## Final comparison — lock the text model

In [ ]:
rows=[('v2 frozen probe (logreg)', PROBE_BASELINE, None),
      ('v3 full fine-tune',        FULL_FT_BASELINE, None),
      ('v4-A frozen + MLP head',   float(np.mean(mlp_f1)),  float(np.std(mlp_f1))),
      ('v4-B LoRA (q,v)',          float(np.mean(lora_f1)), float(np.std(lora_f1)))]
print('3-class severity, patient-level 5-fold CV (macro-F1)\n')
print(f"{'method':<30}{'mean':>8}{'std':>9}")
for n,m,s in rows:
    print(f"{n:<30}{m:>8.3f}{('  --' if s is None else f'{s:>9.3f}')}")

best_name,best_mean,best_std=max(rows,key=lambda r:r[1])
print(f'\nBest: {best_name} ({best_mean:.3f})')
delta=best_mean-PROBE_BASELINE
if best_std is not None and delta < best_std:
    print(f'  NOTE: beats probe by {delta:+.3f}, which is WITHIN one std ({best_std:.3f}). Treat as a wash, not a real gain.')
elif delta<=0:
    print('  NOTE: nothing beat the frozen probe. Lock the probe (0.449) as the honest text result.')
else:
    print(f'  Real gain of {delta:+.3f} over probe, beyond one std. This is the text model.')

summary={'baselines':{'probe':PROBE_BASELINE,'full_ft':FULL_FT_BASELINE},
         'mlp':{'mean':float(np.mean(mlp_f1)),'std':float(np.std(mlp_f1)),'per_fold':[float(x) for x in mlp_f1]},
         'lora':{'mean':float(np.mean(lora_f1)),'std':float(np.std(lora_f1)),'per_fold':[float(x) for x in lora_f1]},
         'best':best_name}
json.dump(summary,open(OUT_DIR/'v4_results.json','w'),indent=2,ensure_ascii=False)
np.savez_compressed(OUT_DIR/'v4_oof.npz',ids=ids,y_true=y,oof_mlp=oof_mlp,oof_lora=oof_lora)
print('\nsaved to',OUT_DIR)

In [ ]:
# per-class report for whichever gentle model won
best_oof = oof_mlp if np.mean(mlp_f1)>=np.mean(lora_f1) else oof_lora
print(classification_report(y, best_oof,
      target_names=['none_minimal','mild_moderate','severe'], digits=3, zero_division=0))

## Decision rule (built into the cell above)